In [23]:
import numpy as np
import pandas as pd

In [24]:
df=pd.read_csv('Bengaluru_House_Data.csv')
df

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00
...,...,...,...,...,...,...,...,...,...
13315,Built-up Area,Ready To Move,Whitefield,5 Bedroom,ArsiaEx,3453,4.0,0.0,231.00
13316,Super built-up Area,Ready To Move,Richards Town,4 BHK,NaN,3600,5.0,NaN,400.00
13317,Built-up Area,Ready To Move,Raja Rajeshwari Nagar,2 BHK,Mahla T,1141,2.0,1.0,60.00
13318,Super built-up Area,18-Jun,Padmanabhanagar,4 BHK,SollyCl,4689,4.0,1.0,488.00


In [25]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  object 
 1   availability  13320 non-null  object 
 2   location      13319 non-null  object 
 3   size          13304 non-null  object 
 4   society       7818 non-null   object 
 5   total_sqft    13320 non-null  object 
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), object(6)
memory usage: 936.7+ KB


In [26]:
for column in df.columns:
    print(df[column].value_counts())
    print("*"*20)

area_type
Super built-up  Area    8790
Built-up  Area          2418
Plot  Area              2025
Carpet  Area              87
Name: count, dtype: int64
********************
availability
Ready To Move    10581
18-Dec             307
18-May             295
18-Apr             271
18-Aug             200
                 ...  
15-Dec               1
16-Oct               1
16-Jan               1
14-Nov               1
14-Jul               1
Name: count, Length: 81, dtype: int64
********************
location
Whitefield                 540
Sarjapur  Road             399
Electronic City            302
Kanakpura Road             273
Thanisandra                234
                          ... 
Park View Layout             1
Xavier Layout                1
Air View Colony              1
akshaya nagar t c palya      1
mvj engineering college      1
Name: count, Length: 1305, dtype: int64
********************
size
2 BHK         5199
3 BHK         4310
4 Bedroom      826
4 BHK          591
3 Bedroom 

In [27]:
df.isna().sum()

area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64

In [28]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df)
profile.to_file('DataAnalysis.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 9/9 [00:00<00:00, 49.04it/s]
/home/kernel00/miniconda3/envs/AI/lib/python3.11/site-packages/seaborn/matrix.py:260: FutureWarning: Format strings passed to MaskedConstant are ignored, but in future may error or produce different behavior
  annotation = ("{:" + self.fmt + "}").format(val)
/home/kernel00/miniconda3/envs/AI/lib/python3.11/site-packages/ydata_profiling/model/missing.py:132: UserWarning: There was an attempt to generate the Heatmap missing values diagrams, but this failed.
        To hide this warning, disable the calculation
        (using `df.profile_report(missing_diagrams={"Heatmap": False}`)
        If this is problematic for your use case, please report this as an issue:
        https://github.com/ydataai/ydata-profiling/issues
        (include the error message: 'could not convert string to float: '--'')
  warnings.warn(


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

## Droppping irrelevent columns

In [29]:
# society has 5502 (41.3%) missing values and also has 2688 distinct value so droppping it

df=df.drop(columns=['society'])

In [30]:
# area typep has 0.518 corealtion with size so area type is redundent also area type is weakly corelated with price (0.107) so dropping area type

# also balcony is weakly corealted with price (0.042) and balcony has 0.299 corealation with size. Thus size can explain area type and balcony to some extent so removing both of the redundent values as they both contribute very less to the price.
df=df.drop(columns=['area_type','balcony'])

In [31]:
# df['size'] has inconsistent data like 2 BHK and 2 bedroom. so extracting just digit out of it and finding out it's corelation with the price column.

df['bhk'] = df['size'].str.extract(r'(\d+)').astype(float)
df['bhk'].corr(df['price'])
df.drop(columns=['size'],inplace=True)

# The r prefix makes it a raw string
# ( )	A capture group. It marks the part of the match you want back. extract returns only what’s inside the parentheses, and raises an error if there are none.
# \d means a digit and + means one or more

In [32]:
df['availability'].value_counts()

availability
Ready To Move    10581
18-Dec             307
18-May             295
18-Apr             271
18-Aug             200
                 ...  
15-Dec               1
16-Oct               1
16-Jan               1
14-Nov               1
14-Jul               1
Name: count, Length: 81, dtype: int64

In [33]:
# This column has class imbalance (79%).A column that's 79% one value carries very little information (low entropy).
# a listing available "18-Dec" vs "18-Jan" isn't obviously going to be priced differently — this isn't like premium societies commanding higher prices. however the flat that are read to move may have higher price.

In [34]:
# before dropping this column,let's built 'ready to move' column and let's see its corelation with the price 
df['ready_to_move'] = (df['availability'] == 'Ready To Move').astype(int)
df['ready_to_move'].corr(df['price'])
df.drop(columns=['ready_to_move'],inplace=True)

In [35]:
#  0.034 is essentially no relationship, even weaker than balcony's 0.042. Dropping it is well justified — and unlike society, this isn't a "practical tradeoff despite real signal" situation. There's just no meaningful relationship here to lose.

df.drop(columns=['availability'],inplace=True)

In [36]:
df.describe()

,bath,price,bhk
count,13247.000000,13320.000000,13304.000000
mean,2.692610,112.565627,2.803743
std,1.341458,148.971674,1.294974
min,1.000000,8.000000,1.000000
25%,2.000000,50.000000,2.000000
50%,2.000000,72.000000,3.000000
75%,3.000000,120.000000,3.000000
max,40.000000,3600.000000,43.000000


## Filling missing values

In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   location    13319 non-null  object 
 1   total_sqft  13320 non-null  object 
 2   bath        13247 non-null  float64
 3   price       13320 non-null  float64
 4   bhk         13304 non-null  float64
dtypes: float64(3), object(2)
memory usage: 520.4+ KB


In [38]:
df=df.dropna(subset=['location'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13319 entries, 0 to 13319
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   location    13319 non-null  object 
 1   total_sqft  13319 non-null  object 
 2   bath        13246 non-null  float64
 3   price       13319 non-null  float64
 4   bhk         13303 non-null  float64
dtypes: float64(3), object(2)
memory usage: 624.3+ KB


In [39]:
# since bath and bhk are int values than the float so median is a good option

#since 0.874 is the corealation between bath and bhk(previously size) so can use bath to fill the missing bhk values.
df['bhk']=df['bhk'].fillna(df.groupby(df['bath'])['bhk'].transform('median'))

In [ ]:

df['bhk'].isna().sum()

16

In [41]:
bhk_bath_null_rows=df[df['bhk'].isna() & df['bath'].isna()]
bhk_bath_null_rows

,location,total_sqft,bath,price,bhk
579,Sarjapur Road,1200 - 2400,NaN,34.185,NaN
1775,IVC Road,2000 - 5634,NaN,124.000,NaN
2264,Banashankari,2400,NaN,460.000,NaN
2809,Sarjapur Road,1200 - 2400,NaN,28.785,NaN
2862,Devanahalli,1500 - 2400,NaN,46.800,NaN
5333,Devanahalli,2100 - 5405,NaN,177.115,NaN
6423,Whitefield,2324,NaN,26.730,NaN
6636,Jigani,1500,NaN,25.490,NaN
6719,Hoskote,800 - 2660,NaN,28.545,NaN
7680,Kasavanhalli,5000,NaN,400.000,NaN


In [42]:
# 16 rows with bhk(size) as nan value also has nan bath value. so filling nan by universal median value  

df['bhk']=df['bhk'].fillna(df['bhk'].median())
df['bhk'].isna().sum()

0

In [43]:
# now using bhk values to fill 73 na values of bath. bath will be replace by the median of those columns which has same bhk values.

df['bath']=df['bath'].fillna(df.groupby('bhk')['bath'].transform('median'))
df['bath'].isna().sum()

0

In [44]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13319 entries, 0 to 13319
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   location    13319 non-null  object 
 1   total_sqft  13319 non-null  object 
 2   bath        13319 non-null  float64
 3   price       13319 non-null  float64
 4   bhk         13319 non-null  float64
dtypes: float64(3), object(2)
memory usage: 624.3+ KB


## further analysis
9 min in video